# 🧩 Паттерны проектирования в Python: Singleton, Factory, Strategy

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">Адаптация классических паттернов под Python</h2>
  <p>Паттерны проектирования — проверенные решения типовых задач. Но в Python многие из них выглядят иначе, чем в C++ или Java. В этой лекции вы узнаете:</p>
  <ul>
    <li>Как реализовать <strong>Singleton</strong> по‑питоньи (метакласс, декоратор, модуль)</li>
    <li>Как создать <strong>Фабрику</strong> без кучи классов (словари, callable, регистрация)</li>
    <li>Как применить <strong>Стратегию</strong> с помощью функций и объектов</li>
  </ul>
</div>

## 🎯 Цели лекции

- Понять, зачем нужны паттерны и когда они полезны в Python
- Реализовать Singleton тремя способами и оценить их плюсы/минусы
- Освоить Factory Method и Abstract Factory в идиоматическом Python
- Научиться использовать Strategy с функциями, лямбдами и объектами-стратегиями
- Увидеть, как динамическая природа Python упрощает паттерны

## 📚 План

1. **Паттерны проектирования: зачем и когда**
   - Что такое паттерн
   - Особенности Python (утиная типизация, функции как объекты, метаклассы)

2. **Singleton (Одиночка)**
   - Классическая проблема: глобальный объект
   - Реализация через `__new__`
   - Реализация через метакласс
   - Модульный Singleton (питоний способ)
   - Когда Singleton — антипаттерн

3. **Factory (Фабрика)**
   - Простая фабрика (функция)
   - Factory Method через абстрактный класс (не Pythonic)
   - Карта типов (словарь классов)
   - Регистрация подклассов через декоратор
   - Abstract Factory на примере семейств виджетов

4. **Strategy (Стратегия)**
   - Проблема: много условных операторов
   - Реализация через интерфейс (традиционно)
   - Pythonic: функции и лямбды как стратегии
   - Выбор стратегии на лету
   - Комбинирование с Factory

5. **Сравнение и итоги**
   - Когда какой паттерн применять
   - Альтернативы (простое наследование, композиция)

6. **Домашнее задание**

## 1. Паттерны проектирования: зачем и когда

Паттерны проектирования — это не готовый код, а **идея** решения повторяющейся проблемы. В статически типизированных языках паттерны часто нужны, чтобы обойти ограничения языка. Python динамический и гибкий, поэтому некоторые паттерны становятся тривиальными или даже не нужными.

### Особенности Python, влияющие на паттерны:

- **Утиная типизация** — не нужны интерфейсы в явном виде
- **Функции — объекты первого класса** — можно передавать и хранить функции как стратегии
- **Метаклассы** — мощный инструмент для изменения поведения классов
- **Модули — синглтоны по умолчанию** (модуль импортируется один раз)
- **Декораторы** — удобны для регистрации стратегий или фабрик

Тем не менее, знание классических паттернов полезно для общения с коллегами и понимания чужого кода.

## 2. Singleton (Одиночка)

**Задача:** гарантировать, что у класса есть только один экземпляр, и предоставить к нему глобальную точку доступа.

### Способ 1: через `__new__` (самый понятный)

In [ ]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value=None):
        # Инициализируем только один раз
        if not hasattr(self, 'initialized'):
            self.value = value
            self.initialized = True

# Проверка
a = Singleton(10)
b = Singleton(20)
print(a is b)        # True
print(a.value, b.value)  # 10, 10

### Способ 2: через метакласс (переиспользуемый)

In [ ]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            instance = super().__call__(*args, **kwargs)
            cls._instances[cls] = instance
        return cls._instances[cls]

class Config(metaclass=SingletonMeta):
    def __init__(self, settings=None):
        self.settings = settings or {}

c1 = Config({"debug": True})
c2 = Config()
print(c1 is c2)       # True
print(c1.settings)    # {"debug": True} (инициализация только при первом вызове)

### Способ 3: модульный Singleton (питоний)
Модули в Python импортируются один раз, поэтому переменная на уровне модуля уже является синглтоном.

In [ ]:
# db.py
class Database:
    def __init__(self):
        self.connected = False

db = Database()   # единственный экземпляр

# main.py
# from db import db   # везде один и тот же объект

**Когда Singleton антипаттерн?**
- Если вы просто хотите избежать глобальных переменных — вы создаёте скрытую глобальную переменную.
- Мешает тестированию (глобальное состояние).
- В Python часто можно обойтись модулем или обычным экземпляром, передаваемым по依赖.

**Рекомендация:** используйте модульный синглтон или внедрение зависимости (dependency injection).

## 3. Factory (Фабрика)

**Задача:** создание объектов без жёсткой привязки к конкретным классам.

### Простая фабрика (функция)

In [ ]:
class Dog:
    def speak(self): return "Woof"

class Cat:
    def speak(self): return "Meow"

def animal_factory(animal_type):
    if animal_type == "dog":
        return Dog()
    elif animal_type == "cat":
        return Cat()
    else:
        raise ValueError(f"Unknown animal: {animal_type}")

animal = animal_factory("dog")
print(animal.speak())

### Factory Method через ассоциативный массив

In [ ]:
class Animal:
    pass

class Dog(Animal):
    def speak(self): return "Woof"

class Cat(Animal):
    def speak(self): return "Meow"

class Bird(Animal):
    def speak(self): return "Chirp"

ANIMALS = {
    "dog": Dog,
    "cat": Cat,
    "bird": Bird,
}

def better_factory(name):
    cls = ANIMALS.get(name)
    if cls is None:
        raise ValueError(f"No such animal: {name}")
    return cls()

animal = better_factory("bird")
print(animal.speak())

### Фабрика с регистрацией через декоратор (расширяемая)

In [ ]:
class Shape:
    pass

shape_registry = {}

def register_shape(name):
    def decorator(cls):
        shape_registry[name] = cls
        return cls
    return decorator

@register_shape("circle")
class Circle(Shape):
    def draw(self): return "Drawing Circle"

@register_shape("square")
class Square(Shape):
    def draw(self): return "Drawing Square"

def shape_factory(name, *args, **kwargs):
    cls = shape_registry.get(name)
    if not cls:
        raise ValueError(f"Shape {name} not registered")
    return cls(*args, **kwargs)

c = shape_factory("circle")
print(c.draw())

### Abstract Factory (семейство продуктов)
В Python часто достаточно передавать фабричные функции в конструктор.

In [ ]:
class LinuxButton:
    def paint(self): return "Linux style button"

class WindowsButton:
    def paint(self): return "Windows style button"

class LinuxCheckbox:
    def paint(self): return "Linux style checkbox"

class WindowsCheckbox:
    def paint(self): return "Windows style checkbox"

# Фабрика — просто пара функций
def linux_factory():
    return (LinuxButton(), LinuxCheckbox())

def windows_factory():
    return (WindowsButton(), WindowsCheckbox())

def create_ui(factory):
    button, checkbox = factory()
    print(button.paint(), checkbox.paint())

create_ui(linux_factory)
create_ui(windows_factory)

## 4. Strategy (Стратегия)

**Задача:** определить семейство алгоритмов, инкапсулировать каждый из них и сделать их взаимозаменяемыми.

В Python стратегия — это просто функция или объект с методом `execute()`.

### Классическая реализация (интерфейс-класс)

In [ ]:
class SortStrategy:
    def sort(self, data):
        raise NotImplementedError

class BubbleSort(SortStrategy):
    def sort(self, data):
        data = list(data)
        n = len(data)
        for i in range(n-1):
            for j in range(n-i-1):
                if data[j] > data[j+1]:
                    data[j], data[j+1] = data[j+1], data[j]
        return data

class QuickSort(SortStrategy):
    def sort(self, data):
        if len(data) <= 1:
            return data
        pivot = data[0]
        left = [x for x in data[1:] if x < pivot]
        right = [x for x in data[1:] if x >= pivot]
        return self.sort(left) + [pivot] + self.sort(right)

class Context:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy

    def execute_strategy(self, data):
        return self._strategy.sort(data)

ctx = Context(QuickSort())
print(ctx.execute_strategy([5,2,8,1,9]))

### Pythonic: функции как стратегии

In [ ]:
def bubble_sort(data):
    data = list(data)
    n = len(data)
    for i in range(n-1):
        for j in range(n-i-1):
            if data[j] > data[j+1]:
                data[j], data[j+1] = data[j+1], data[j]
    return data

def quick_sort(data):
    if len(data) <= 1:
        return data
    pivot = data[0]
    left = [x for x in data[1:] if x < pivot]
    right = [x for x in data[1:] if x >= pivot]
    return quick_sort(left) + [pivot] + quick_sort(right)

class Sorter:
    def __init__(self, strategy):
        self.strategy = strategy

    def sort(self, data):
        return self.strategy(data)

sorter = Sorter(quick_sort)
print(sorter.sort([5,2,8,1,9]))

# Можно сменить стратегию на лету
sorter.strategy = bubble_sort
print(sorter.sort([5,2,8,1,9]))

### Выбор стратегии из набора (с использованием словаря)

In [ ]:
def apply_discount_standard(price):
    return price * 0.95

def apply_discount_premium(price):
    return price * 0.90

def apply_discount_vip(price):
    return price * 0.80

discounts = {
    "standard": apply_discount_standard,
    "premium": apply_discount_premium,
    "vip": apply_discount_vip,
}

def get_discounted_price(price, discount_type):
    strategy = discounts.get(discount_type, apply_discount_standard)
    return strategy(price)

print(get_discounted_price(100, "vip"))   # 80
print(get_discounted_price(100, "unknown")) # 95

### Комбинация Factory + Strategy

In [ ]:
# Стратегии валидации
def validate_email(email):
    return "@" in email and "." in email.split("@")[-1]

def validate_phone(phone):
    return phone.isdigit() and len(phone) == 10

# Фабрика стратегий
validators = {
    "email": validate_email,
    "phone": validate_phone,
}

def get_validator(field_type):
    return validators.get(field_type, lambda x: False)

class FormField:
    def __init__(self, name, field_type):
        self.name = name
        self.validator = get_validator(field_type)

    def is_valid(self, value):
        return self.validator(value)

email_field = FormField("user_email", "email")
print(email_field.is_valid("test@example.com"))   # True
print(email_field.is_valid("invalid"))            # False

## 5. Сравнение и итоги

| Паттерн | Классическая реализация | Pythonic реализация |
|---------|------------------------|---------------------|
| Singleton | `__new__` или метакласс | Модуль (файл) |
| Factory | Класс с методом `create` | Словарь функций / классов, регистрация декоратором |
| Strategy | Интерфейсный класс + иерархия | Функция, лямбда, или объект с `__call__` |

### Общие рекомендации

- **Singleton:** если нужен глобальный объект состояния, лучше использовать модуль или внедрение зависимости. Метакласс — когда нужна иерархия синглтонов.
- **Factory:** словарь классов — самый гибкий и понятный способ. Декоратор для регистрации удобен, когда классов много и они добавляются динамически.
- **Strategy:** функции — отличная замена многочисленным классам. Если стратегия сложная (хранит состояние), тогда используйте класс.

### 🧠 Проверь себя

1. Почему модуль в Python является синглтоном?  
2. Как добавить новую стратегию в словарь `discounts` без изменения существующего кода?  
3. В чём преимущество фабрики перед прямым вызовом конструктора?

<details>
<summary>Ответы</summary>

1. Модуль импортируется один раз, и все импортёры получают один и тот же объект модуля.  
2. Достаточно добавить новую функцию (стратегию) и записать её в словарь: `discounts["new"] = new_strategy`.  
3. Фабрика централизует логику создания, позволяет вернуть разные типы объектов в зависимости от параметров, упрощает тестирование и замену реализации.

</details>

---

## 📖 Домашнее задание

1. **Singleton с ленивой загрузкой** – реализуйте синглтон, который создаёт экземпляр только при первом обращении к атрибуту или методу (не через `__new__`, а через дескриптор или `__getattr__`).

2. **Фабрика фигур с поддержкой параметров** – расширьте фабрику фигур, чтобы можно было создавать `Circle(radius)`, `Rectangle(width, height)`. Используйте словарь, который хранит не класс, а функцию-конструктор, принимающую аргументы.

3. **Стратегия расчета доставки** – напишите класс `Order` с методом `calculate_shipping`, который использует стратегию. Реализуйте стратегии: `StandardShipping` (фиксированная цена 5), `ExpressShipping` (10 + 0.5 за каждый кг), `Pickup` (бесплатно). Используйте функции в качестве стратегий.